# Task 3
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [1]:
import numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader, TensorDataset
import os
import torch
from torchvision import transforms
from torchvision.models import efficientnet_v2_s,EfficientNet_V2_S_Weights
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.datasets as datasets
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sn
import pandas as pd
import random
# Add any other imports you need here

In [2]:
# The device is automatically set to GPU if available, otherwise CPU
# If you want to force the device to CPU, you can change the line to
# device = torch.device("cpu")
# When using the GPU, it is important that your model and all data are on the 
# same device.
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [3]:
"""
Transform, resize and normalize the images and then use a pretrained model to extract 
the embeddings.
"""
# TODO: define a transform to pre-process the images
# The required pre-processing depends on the pre-trained model you choose 
# below. 
# See https://pytorch.org/vision/stable/models.html#using-the-pre-trained-models

#source of inspiration/detail on how to use various functions: 
# https://github.com/mortezamg63/Accessing-and-modifying-different-layers-of-a-pretrained-model-in-pytorch/blob/master/README.md

"""train_transforms = transforms.Compose([
    transforms.Resize(232),                      
    transforms.CenterCrop(224),                  
    transforms.ToTensor(),                       
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  
])"""
#train_transforms = transforms.Compose([transforms.ToTensor()])

#EfficientNet seems to be better than Resnet18
weights = EfficientNet_V2_S_Weights.DEFAULT
model = efficientnet_v2_s(weights=weights).to(device)
train_transforms= weights.transforms() #automatically gets transforms associated with weights
print(weights.transforms())

train_dataset = datasets.ImageFolder(root="dataset/", transform=train_transforms)

#plt.imshow(np_train)
#plt.show()
# Hint: adjust batch_size and num_workers to your PC configuration, so that you don't 
# run out of memory (VRAM if on GPU, RAM if on CPU)
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=100,
                          shuffle=False, #was false, I set to True
                          pin_memory=True, num_workers=4) #was num_workers = 16

# TODO: define a model for extraction of the embeddings (Hint: load a pretrained model,
# more info here: https://pytorch.org/vision/stable/models.html)
#print(model)
model_no_last_layer = nn.Sequential(*list(model.children())[:-1])
#print(model_no_last_layer)
embedding_size = 1280 # Dummy variable, replace with the actual embedding size once you pick your model

num_images = len(train_dataset)
embeddings = []
#embeddings = np.zeros((num_images, embedding_size))

# TODO: Use the model to extract the embeddings. Hint: remove the last layers of the 
# model to access the embeddings the model generates. 
model_no_last_layer.eval() #sets to evaluation mode
print("starting")
count = 0
for inputs,labels in train_loader:
    inputs= inputs.to(device) #send data to GPU as well
    with torch.no_grad():
        output = model_no_last_layer(inputs)
        output_np = output.cpu().detach().squeeze().numpy()  
    print("Output shape:", output_np.shape)
    embeddings.extend(output_np)
    count += 100
    print("Count: ",count)
np.save('dataset/embeddings.npy', embeddings)

ImageClassification(
    crop_size=[384]
    resize_size=[384]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
starting
Output shape: (100, 1280)
Count:  100
Output shape: (100, 1280)
Count:  200
Output shape: (100, 1280)
Count:  300
Output shape: (100, 1280)
Count:  400
Output shape: (100, 1280)
Count:  500
Output shape: (100, 1280)
Count:  600
Output shape: (100, 1280)
Count:  700
Output shape: (100, 1280)
Count:  800
Output shape: (100, 1280)
Count:  900
Output shape: (100, 1280)
Count:  1000
Output shape: (100, 1280)
Count:  1100
Output shape: (100, 1280)
Count:  1200
Output shape: (100, 1280)
Count:  1300
Output shape: (100, 1280)
Count:  1400
Output shape: (100, 1280)
Count:  1500
Output shape: (100, 1280)
Count:  1600
Output shape: (100, 1280)
Count:  1700
Output shape: (100, 1280)
Count:  1800
Output shape: (100, 1280)
Count:  1900
Output shape: (100, 1280)
Count:  2000
Output shape: (100, 1280)
Count:  2100
Output sha

In [4]:
#original get data
def get_data(file, train=True):
    """
    Load the triplets from the file and generate the features and labels.

    input: file: string, the path to the file containing the triplets
          train: boolean, whether the data is for training or testing

    output: X: numpy array, the features
            y: numpy array, the labels
    """
    triplets = []
    with open(file) as f:
        for line in f:
            triplets.append(line)

    # generate training data from triplets
    train_dataset = datasets.ImageFolder(root="dataset/",
                                         transform=None)
    filenames = [s[0].split('\\')[-1].replace('.jpg', '') for s in train_dataset.samples]
    embeddings = np.load('dataset/embeddings.npy')
    # TODO: Normalize the embeddings
    embeddings = embeddings / (np.linalg.norm(embeddings,axis =1).reshape(-1,1)) #just divide by norm
    
    file_to_embedding = {}
    for i in range(len(filenames)):
        file_to_embedding[filenames[i]] = embeddings[i]
    X = []
    y = []
    # use the individual embeddings to generate the features and labels for triplets
    for t in triplets:
        emb = [file_to_embedding[a] for a in t.split()]
        X.append(np.hstack([emb[0], emb[1], emb[2]]))
        y.append(1)
        # Generating negative samples (data augmentation)
        if train:
            X.append(np.hstack([emb[0], emb[2], emb[1]]))
            y.append(0)
    X = np.vstack(X)
    y = np.hstack(y)
    return X, y

In [5]:
def get_train_data(file):
    #Load the triplets from the file and generate the features and labels.

    #input: file: string, the path to the file containing the triplets
    #      train: boolean, whether the data is for training or testing

    #output: X: numpy array, the features
    #        y: numpy array, the labels
    triplets = []
    with open(file) as f:
        for line in f:
            triplets.append(line)
    # generate training data from triplets
    train_dataset = datasets.ImageFolder(root="dataset/",
                                         transform=None)
    filenames = [s[0].split('\\')[-1].replace('.jpg', '') for s in train_dataset.samples]
    embeddings = np.load('dataset/embeddings.npy')
    # TODO: Normalize the embeddings
    embeddings = embeddings / (np.linalg.norm(embeddings,axis =1).reshape(-1,1)) #just divide by norm
    
    file_to_embedding = {}
    #print(len(filenames))
    print(embeddings.shape)
    for i in range(len(filenames)):
        file_to_embedding[filenames[i]] = embeddings[i]
    # use the individual embeddings to generate the features and labels for triplets

    train_X = []
    train_y = []
    test_X = []
    test_y = []
    for trip in triplets:
        emb = [file_to_embedding[a] for a in trip.split()]
        count = sum([0 if int(t)<=4700 else 1 for t in trip.split()])
        #count = 3 if all triplet members > 4700
        if (count == 0):
            train_X.append(np.hstack([emb[0], emb[1], emb[2]]))
            train_y.append(1)
            # Generating negative samples (data augmentation)
            train_X.append(np.hstack([emb[0], emb[2], emb[1]]))
            train_y.append(0)
        else:
            test_X.append(np.hstack([emb[0], emb[1], emb[2]]))
            test_y.append(1)
            # Generating negative samples (data augmentation)
            test_X.append(np.hstack([emb[0], emb[2], emb[1]]))
            test_y.append(0)
    train_X = np.vstack(train_X)
    train_y = np.hstack(train_y)
    test_X = np.vstack(test_X)
    test_y = np.hstack(test_y)
    return train_X,train_y,test_X,test_y

Hint: adjust batch_size and num_workers to your PC configuration, so that you don't run out of memory (VRAM if on GPU, RAM if on CPU)

In [6]:
def create_loader_from_np(X, y = None, train = True, batch_size=1000, shuffle=True, num_workers = 4):
    """
    Create a torch.utils.data.DataLoader object from numpy arrays containing the data.

    input: X: numpy array, the features
           y: numpy array, the labels
    
    output: loader: torch.data.util.DataLoader, the object containing the data
    """
    if train:
        # Attention: If you get type errors you can modify the type of the
        # labels here
        dataset = TensorDataset(torch.from_numpy(X).type(torch.float), 
                                torch.from_numpy(y).type(torch.float)) #was a long
    else:
        dataset = TensorDataset(torch.from_numpy(X).type(torch.float))
    loader = DataLoader(dataset=dataset,
                        batch_size=batch_size,
                        shuffle=shuffle,
                        pin_memory=True, num_workers=num_workers)
    return loader

TODO: define a model. Here, the basic structure is defined, but you need to fill in the details

#### Notes
- SiLU worse than ReLU
- efficientNet is better than ResNet for creating the embeddings
- SGD optimizer seems to perform worse than Adam
- adding more layers seems to overfit and causes worse performance
- 128 batch size seems good
- learning rate depends on model

In [22]:
class NetReLU(nn.Module):
    """
    The model class, which defines our classifier.
    """
    def __init__(self):
        """
        The constructor of the model.
        """
        super().__init__()
        #use sequential to build model, if need to access layers can change later
        """According to Google:
            - Dropout after any non-output Layer
            - good values for dropout are 0.2 for input and 0.5 for hidden
            - Debate about where batch normalization (probably after activation)
        """
        self.layers = nn.Sequential(
            nn.Linear(3*1280,512),
            nn.Dropout(0.7), #dropout as regularization is probably a good idea to try and avoid overfitting 
            nn.BatchNorm1d(512), #according to google generally increases performance of NN
            nn.ReLU(),        #is fairly commonly used, SiLU is worse      
            nn.Linear(512,64),
            nn.Dropout(0.5), 
            nn.BatchNorm1d(64), 
            nn.ReLU(),       
            nn.Linear(64,1),
            nn.Sigmoid()) #to have values between 0 and 1

    def forward(self, x):
        """
        The forward pass of the model.

        input: x: torch.Tensor, the input to the model

        output: x: torch.Tensor, the output of the model
        """
        return self.layers(x)

In [36]:
TRAIN_TRIPLETS = 'train_triplets.txt'
training_batch_size = 128

# Entire Training Data
X,y = get_data(TRAIN_TRIPLETS,train=True)
train_loader = create_loader_from_np(X, y, train = True, batch_size=training_batch_size)

# Training data split into training & validation
X_train, y_train, X_val, y_val = get_train_data(TRAIN_TRIPLETS)
training_loader = create_loader_from_np(X_train, y_train, train=True, batch_size=training_batch_size)
validation_loader = create_loader_from_np(X_val, y_val, train=True, batch_size=training_batch_size)

# delete the loaded training data to save memory, as the data loader copies
del X_train
del y_train
del X_val
del y_val
del X
del y
print("done")

(10000, 1280)
done


In [30]:
TEST_TRIPLETS = 'test_triplets.txt'

# repeat for testing data
X_test, y_test = get_data(TEST_TRIPLETS, train=False)
test_loader = create_loader_from_np(X_test, train = False, batch_size=2000, shuffle=False)
del X_test
del y_test
print("done")

done


In [39]:
"""
The training procedure of the model; it accepts the training data, defines the model 
and then trains it.

input: train_loader: torch.data.util.DataLoader, the object containing the training data
    
compute: model: torch.nn.Module, the trained model
"""
models = [NetReLU()]
# TODO: define a loss function, optimizer and proceed with training. Hint: use the part 
# of the training data as a validation split. After each epoch, compute the loss on the 
# validation split and print it out. This enables you to see how your model is performing 
# on the validation data before submitting the results on the server. After choosing the 
# best model, train it on the whole training data.

#split training & validation data
#train_dataset, valid_dataset = torch.utils.data.random_split(train_loader.dataset, [0.8,0.2])
#training_loader = DataLoader(train_dataset,batch_size = train_loader.batch_size)
#validation_loader = DataLoader(valid_dataset,batch_size = train_loader.batch_size)
train_len = len(training_loader.dataset)
val_len = len(validation_loader.dataset)
print(train_len)
print(val_len)

criterion = nn.BCELoss() #binary cross entropy loss for one output neuron
n_epochs = 500
for model in models: 
    print(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    model.to(device)
    
    for epoch in range(n_epochs):  
        #Training loop
        train_losses = []
        model.train()
        for x_batch, y_batch in training_loader:
            x_batch, y_batch = x_batch.to(device),y_batch.to(device).unsqueeze(1)
            #y_batch = y_batch.unsqueeze(1).detach()
            #print(x_batch.shape,y_batch.shape)
            y_pred = model(x_batch)
            #print(y_pred)
            loss = criterion(y_pred, y_batch)
            train_losses.append(loss.item())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f'epoch: {epoch:2}  Training loss: {loss.mean()}')

        #Validation loop
        val_losses = []
        model.eval()
        count = 0
        for x_batch, y_batch in validation_loader: 
            x_batch, y_batch = x_batch.to(device), y_batch.to(device).unsqueeze(1)
            y_pred = model(x_batch)
            loss = criterion(y_pred, y_batch)
            val_losses.append(loss.item())
            
            y_pred = y_pred.cpu().detach().numpy()
            y_pred[y_pred >= 0.5] = 1
            y_pred[y_pred < 0.5] = 0
            y_batch = y_batch.cpu().detach().numpy()
            #print(y_batch,y_pred)
            for i,pred in enumerate(y_pred):
                #print(i,pred,y_batch[i])
                if(pred != y_batch[i]):
                    count += 1
        print(f'epoch: {epoch:2}  Validation loss: {loss.mean()}')
        print(f'epoch: {epoch:2}  Accuracy: {1-(count/val_len)}')
    del model

99146
19884
NetReLU(
  (layers): Sequential(
    (0): Linear(in_features=3840, out_features=512, bias=True)
    (1): Dropout(p=0.7, inplace=False)
    (2): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=64, bias=True)
    (5): Dropout(p=0.5, inplace=False)
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Linear(in_features=64, out_features=1, bias=True)
    (9): Sigmoid()
  )
)
epoch:  0  Training loss: 0.6031937003135681
epoch:  0  Validation loss: 0.5000552535057068
epoch:  0  Accuracy: 0.7306879903439951
epoch:  1  Training loss: 0.5209035277366638
epoch:  1  Validation loss: 0.5018174648284912
epoch:  1  Accuracy: 0.7335546167773084
epoch:  2  Training loss: 0.5392363667488098
epoch:  2  Validation loss: 0.488850861787796
epoch:  2  Accuracy: 0.7403439951719976
epoch:  3  Training loss: 0.48941799998283386
epoch:  3  Valida

KeyboardInterrupt: 

In [40]:
# Train final model
model = NetReLU()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
model.to(device)
model.train()

for epoch in range(11):  #10 is roughly when the model starts to overfit
    train_losses = []
    for x_batch, y_batch in train_loader: #entire training dataset
        x_batch, y_batch = x_batch.to(device),y_batch.to(device).unsqueeze(1)
        #y_batch = y_batch.unsqueeze(1).detach()
        #print(x_batch.shape,y_batch.shape)
        y_pred = model(x_batch)
        #print(y_pred)
        loss = criterion(y_pred, y_batch)
        train_losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if(epoch%5 ==0):
        print(f'epoch: {epoch:2}  Training loss: {loss.mean()}')
print("done")

epoch:  0  Training loss: 0.5151530504226685
epoch:  5  Training loss: 0.4580303430557251
epoch: 10  Training loss: 0.34382888674736023
done


In [41]:
"""
The testing procedure of the model; it accepts the testing data and the trained model and 
then tests the model on it.

input: model: torch.nn.Module, the trained model
       loader: torch.data.util.DataLoader, the object containing the testing data
        
compute: None, the function saves the predictions to a results.txt file
"""

model.eval()
predictions = []
# Iterate over the test data
with torch.no_grad(): # We don't need to compute gradients for testing
    for [x_batch] in test_loader:
        x_batch= x_batch.to(device)
        predicted = model(x_batch)
        predicted = predicted.cpu().numpy()
        # Rounding the predictions to 0 or 1
        predicted[predicted >= 0.5] = 1
        predicted[predicted < 0.5] = 0
        predictions.append(predicted)
    predictions = np.vstack(predictions)
    print(predictions)
np.savetxt("results.txt", predictions, fmt='%i')
print("Results saved to results.txt")

[[0.]
 [0.]
 [0.]
 ...
 [0.]
 [1.]
 [0.]]
Results saved to results.txt
